# 02. Benchmarking Probability Distributions for Extreme Risk

## Objective
To accurately price rare financial events using Quantum Amplitude Estimation (QAE), the classical probability distribution loaded into the quantum grid must be exceptionally robust.

This notebook evaluates three modeling approaches:
1. **Single Lognormal Model:** Standard in finance, but often decays too fast in the tail.
2. **Single Generalized Pareto Distribution (GPD):** Excellent for extremes, but suffers from a "theoretical blind spot" below its attachment threshold.
3. **The Proposed Spliced Model (Lognormal + GPD):** A hybrid architecture that ensures complete domain coverage and robust tail estimation.

We will discretize these models onto a simulated Quantum Grid ($2^n$ bins) using logarithmic binning and benchmark their **Discretized Expected Excess Loss (EEL)** against the empirical truth.

In [11]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import lognorm, genpareto

from src.distribution_fit import optimize_gpd_threshold_ad, fit_spliced_distribution, spliced_cdf
from src.discretization import get_log_spaced_grid

plt.style.use('seaborn-v0_8-whitegrid')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Data Loading and Automated Threshold Retrieval
We initialize the pipeline by loading the pre-processed empirical dataset. Instead of hardcoding an arbitrary threshold (e.g., 95%), we dynamically calculate the optimal attachment point ($u$) using the Conditional A-D Statistic optimizer developed in Notebook 01...

In [6]:
# Load dataset
data_path = '../data/fema_sample_data.csv'
df = pd.read_csv(data_path)

raw_data = pd.to_numeric(df['projectAmount'], errors='coerce').dropna().values
data = raw_data[raw_data > 1000]

# Automatically find the optimal threshold to maintain the pipeline logic
best_u, best_q, _, _ = optimize_gpd_threshold_ad(data, q_start=0.90, q_end=0.98, step=0.005)

print(f"Total valid samples: {len(data):,}")
print(f"Auto-Calculated Optimal Threshold (u): ${best_u:,.2f} (at {best_q * 100:.1f}% Quantile)")

Total valid samples: 110,632
Auto-Calculated Optimal Threshold (u): $1,919,204.96 (at 94.5% Quantile)


## 2. Model Rationale: The Spliced Architecture

Before calibrating the models, it is crucial to understand **why** a hybrid approach is strictly required for quantum tail-risk pricing.

In the hybrid distribution model, we strategically choose an optimal threshold $u$ to divide the distribution into a **'body'** and a **'tail'**. This architecture accurately describes the heavy-tailed extremes while maintaining a valid, mathematically sound probability density in the lower intervals where a single GPD model has no definition (the probability blind spot).

Selecting this optimal attachment point $u$ is a classic **bias-variance tradeoff**:

- **Low Threshold ($u \downarrow$):** High bias due to non-asymptotic data contamination.
- **High Threshold ($u \uparrow$):** High variance due to sample scarcity.

By utilizing the A-D optimized threshold, we seamlessly splice a Lognormal body with a GPD tail:

$F_{spliced}(x) = F_{LN}(x)$ , if $x \le u$

$F_{spliced}(x) = W + (1 - W) \cdot F_{GPD}(x - u)$ , if $x > u$

**Where:**

- $F_{LN}$ is the Lognormal CDF (representing the distribution body).
- $F_{GPD}$ is the Generalized Pareto CDF (representing the heavy tail).
- $W$ **is the Probability Anchor (CDF at threshold $u$).** It represents the empirical cumulative probability of the body, calculated as $P(X < u)$.

**The Role of $W$ (Probability Conservation):** By introducing the probability anchor $W$ and applying an affine transformation to the GPD tail via $(1 - W)$, we guarantee absolute probability conservation. This ensures the Cumulative Distribution Function (CDF) seamlessly connects at the attachment point $u$, satisfying the physical constraints of probability amplitude encoding in quantum states with zero overlap or leakage.

## 3. Model Calibration
With the theoretical framework established and the optimal threshold $u$ calculated, we now independently calibrate the three probability models (Single Lognormal, Single GPD, and the Spliced Model) to the empirical data...

In [7]:
# ---------------------------------------------------------
# A. Single Lognormal Fit (Full Domain)
# ---------------------------------------------------------
shape_ln, loc_ln, scale_ln = lognorm.fit(data, floc=0)

def single_lognorm_cdf(x):
    return lognorm.cdf(x, s=shape_ln, scale=scale_ln)

# ---------------------------------------------------------
# B. Single GPD Fit (Conditional Tail)
# ---------------------------------------------------------
excesses = data[data > best_u] - best_u
c_gpd, loc_gpd, scale_gpd = genpareto.fit(excesses, floc=0)

def single_gpd_cdf(x):
    # GPD is mathematically undefined below best_u. We force it to 0.
    x = np.asarray(x)
    return np.where(x >= best_u, genpareto.cdf(x - best_u, c=c_gpd, scale=scale_gpd), 0.0)

# ---------------------------------------------------------
# C. Spliced Model Fit (Lognormal + GPD)
# ---------------------------------------------------------
w, _, ln_params_splice, gpd_params_splice = fit_spliced_distribution(data, threshold=best_u)

def hybrid_spliced_cdf(x):
    return spliced_cdf(x, w, best_u, ln_params_splice, gpd_params_splice)

print("All three continuous probability models successfully calibrated.")

All three continuous probability models successfully calibrated.


## 4. Quantum Discretization Error Analysis
To load a continuous distribution into a NISQ-era quantum computer, we map the probabilities onto a discrete grid of $N = 2^n$ states. 

Using exactly **$n = 4$ qubits (16 bins)** and a logarithmic spatial grid, we calculate the quantum-discretized Expected Excess Loss (EEL) for each model across different VaR evaluations (90%, Optimal, 99%).ds.

In [10]:
# ---------------------------------------------------------
# Quantum Grid Constraints
# ---------------------------------------------------------
N_QUBITS = 4                     # 2^4 = 16 quantum states
X_MIN = np.percentile(data, 1.0) # Lower bound cut-off (1st percentile)
X_MAX = np.max(data) * 3         # Upper truncation bound (Black Swan simulation)

# Evaluation Thresholds: 90%, the optimal AD threshold, and 99% deep tail
var_levels = [0.90, best_q, 0.95,0.99]

for alpha in var_levels:
    M = np.percentile(data, alpha * 100)
    
    # 1. Empirical Truth
    empirical_excesses = data[data > M] - M
    true_eel = np.sum(empirical_excesses) / len(data)
    
    # 2. Map continuous CDFs onto the Quantum Grid (16 midpoints & probabilities)
    x_mids_ln, p_mass_ln = get_log_spaced_grid(single_lognorm_cdf, X_MIN, X_MAX, N_QUBITS)
    x_mids_gpd, p_mass_gpd = get_log_spaced_grid(single_gpd_cdf, X_MIN, X_MAX, N_QUBITS)
    x_mids_spliced, p_mass_spliced = get_log_spaced_grid(hybrid_spliced_cdf, X_MIN, X_MAX, N_QUBITS)
    
    # 3. Calculate Discretized Quantum EEL (Sum of Prob * Payoff)
    eel_ln = np.sum(p_mass_ln * np.maximum(0, x_mids_ln - M))
    eel_gpd = np.sum(p_mass_gpd * np.maximum(0, x_mids_gpd - M))
    eel_spliced = np.sum(p_mass_spliced * np.maximum(0, x_mids_spliced - M))
    
    print("=" * 85)
    print(f"Evaluation Lower Bound: {alpha*100:.1f}% VaR (Threshold M = ${M:,.2f})")
    print("-" * 85)
    print(f"{'Probability Model':<28} | {'Quantum EEL (4 Qubits)':<25} | {'Relative Error':<15}")
    print("-" * 85)
    print(f"{'Empirical (True Benchmark)':<28} | ${true_eel:<24.2f} | 0.00%")
    print(f"{'A. Single Lognormal':<28} | ${eel_ln:<24.2f} | {(eel_ln/true_eel - 1)*100:>7.2f}%")
    
    # Highlight GPD blindspot if evaluated below the POT threshold
    if alpha < best_q:
        err_str = f"{(eel_gpd/true_eel - 1)*100:>7.2f}% (Blindspot)"
    else:
        err_str = f"{(eel_gpd/true_eel - 1)*100:>7.2f}%"
        
    print(f"{'B. Single GPD (POT)':<28} | ${eel_gpd:<24.2f} | {err_str}")
    print(f"{'C. Spliced Model':<28} | ${eel_spliced:<24.2f} | {(eel_spliced/true_eel - 1)*100:>7.2f}%")
print("=" * 85)

Evaluation Lower Bound: 90.0% VaR (Threshold M = $825,225.14)
-------------------------------------------------------------------------------------
Probability Model            | Quantum EEL (4 Qubits)    | Relative Error 
-------------------------------------------------------------------------------------
Empirical (True Benchmark)   | $952959.49                | 0.00%
A. Single Lognormal          | $229075.94                |  -75.96%
B. Single GPD (POT)          | $22762273.37              | 2288.59% (Blindspot)
C. Spliced Model             | $1310812.70               |   37.55%
Evaluation Lower Bound: 94.5% VaR (Threshold M = $1,919,204.96)
-------------------------------------------------------------------------------------
Probability Model            | Quantum EEL (4 Qubits)    | Relative Error 
-------------------------------------------------------------------------------------
Empirical (True Benchmark)   | $874461.65                | 0.00%
A. Single Lognormal          | $15

## Conclusion
As demonstrated above using a 4-qubit quantum grid simulation:
* **Single Lognormal** consistently exhibits severe negative errors (often >70% underestimation), failing to allocate sufficient quantum amplitudes to "Black Swan" states.
* **Single GPD** performs well in the deep tail, but structurally fails at lower thresholds (massive positive error marked as `Blindspot`) because it lacks probability definition below the optimal attachment point.
* **The Spliced Model** elegantly bridges this gap, providing the most stable and robust probability matrix for the subsequent Quantum Amplitude Estimation circuits.its.